# Notebook Setup & Imports

In [1]:
import sys
from pathlib import Path

# Add project root to PYTHONPATH
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

import torch
import pandas as pd
import numpy as np

# Import Project Modules

In [2]:
from src.dataset import load_data, make_data_loader, make_windows
from src.model import StockMLP, StockCNN
from src.train import train
from src.evaluate import evaluate

# Configuration

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "train.csv"
BATCH_SIZE = 512
WINDOW_SIZE = 30
EPOCHS = 10
USE_CNN = False  # switch between MLP and CNN
NORMALIZE = True

# Load Dataset

In [4]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Please place train.csv inside data/ directory."
    )

df = load_data(DATA_PATH, frac=0.1, random_state=42)
print(f"Rows loaded: {len(df):,}")
# load_data already samples the rows
df.describe()


Rows loaded: 2,103,352


,Date,Open,High,Low,Close,Volume
count,2103352,2.103352e+06,2.103352e+06,2.103352e+06,2.103352e+06,2.103352e+06
mean,2023-11-20 17:48:34.378953,3.281022e+01,3.326131e+01,3.236123e+01,3.280755e+01,1.274352e+06
min,2023-01-13 00:00:00,1.000000e-02,7.900000e-02,1.000000e-02,7.851000e-02,0.000000e+00
25%,2023-06-21 00:00:00,6.250000e+00,6.410000e+00,6.100000e+00,6.250000e+00,4.520000e+04
50%,2023-11-20 00:00:00,1.574000e+01,1.599000e+01,1.549000e+01,1.573000e+01,2.336000e+05
75%,2024-04-23 00:00:00,3.592000e+01,3.645000e+01,3.537000e+01,3.591000e+01,9.069000e+05
max,2024-09-23 00:00:00,4.293400e+02,5.754100e+02,4.220000e+02,4.288500e+02,5.617574e+08
std,NaN,4.740298e+01,4.794830e+01,4.686489e+01,4.741497e+01,4.880477e+06


# Creating Sliding Windows

In [5]:
X, y = make_windows(df, window=WINDOW_SIZE)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Positive ratio:", y.mean())


X shape: (2100592, 30)
y shape: (2100592,)
Positive ratio: 0.5001394844881824


# Train / Validation Split

In [6]:
split_idx = int(len(X) * 0.8)

X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print(f"Total samples: {len(X):,}")
print(f"Train samples: {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")


Total samples: 2,100,592
Train samples: 1,680,473
Validation samples: 420,119


# DataLoaders

In [7]:
from torch.utils.data import TensorDataset, DataLoader

if NORMALIZE:
    from src.features import normalize
    X_train = normalize(X_train)
    X_val = normalize(X_val)

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
)

val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# Initialize Model

In [8]:
if USE_CNN:
    model = StockCNN()
    print("Using CNN model")
else:
    model = StockMLP(input_dim=WINDOW_SIZE)
    print("Using MLP model")

model

Using MLP model


StockMLP(
  (net): Sequential(
    (0): Linear(in_features=30, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

# Train Model

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCEWithLogitsLoss()
train(
    model=model,
    loader=train_loader,
    epochs=EPOCHS,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
)


hello
Epoch 1/10 - Loss: 0.5924
Epoch 2/10 - Loss: 0.5421
Epoch 3/10 - Loss: 0.5316
Epoch 4/10 - Loss: 0.5283
Epoch 5/10 - Loss: 0.5262
Epoch 6/10 - Loss: 0.5241
Epoch 7/10 - Loss: 0.5225
Epoch 8/10 - Loss: 0.5209
Epoch 9/10 - Loss: 0.5200
Epoch 10/10 - Loss: 0.5194


# Evaluate on Validation Set

In [10]:
model.to(device)

val_accuracy = evaluate(
    model,
    X_val,
    y_val,
    device=device,
    batch_size=512,
)

print(f"Validation Accuracy: {val_accuracy:.4f}")


Validation Accuracy: 0.7447


# Save Trained Model

In [11]:
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

model_path = MODELS_DIR / "stock_model_v1.pt"
torch.save(model.state_dict(), model_path)

print(f"Model saved to: {model_path}")

Model saved to: /home/mega/projects/forth-year/NN/stock-trend/models/stock_model_v1.pt


# Quick Sanity Prediction

In [12]:
model.eval()

sample = torch.tensor(X_val[:5], dtype=torch.float32).to(device)
with torch.no_grad():
    logits = model(sample)
    probs = torch.sigmoid(logits)

print("Predicted probabilities:", probs.cpu().numpy())
print("True labels:", y_val[:5])

Predicted probabilities: [0.32243353 0.39118978 0.4777463  0.82514304 0.18868232]
True labels: [0 0 0 1 0]
